In [ ]:
import os, sys, subprocess
print("python:", sys.version.split()[0])

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "einops",
                "safetensors", "opencv-python-headless", "trimesh", "scikit-image", "pyproj"])
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "onnxruntime"])
if r.returncode != 0:
    print("ultralytics/onnxruntime install failed; will fall back to shadow-only masks")

ROOT = "/content"; OUT = os.path.join(ROOT, "out")
os.makedirs(OUT, exist_ok=True)
os.makedirs(os.path.join(OUT, "ckpt"), exist_ok=True)
if not os.path.exists("/content/third_party/vggt_omega"):
    !git clone -q https://github.com/facebookresearch/vggt-omega.git /content/third_party/vggt_omega

try:
    import ultralytics
    print("ultralytics ready:", ultralytics.__version__)
except ImportError:
    print("ultralytics unavailable -> shadow-only masks (acceptable for demo)")
print("setup ok")

python: 3.13.15
ultralytics ready: 8.4.146
setup ok


In [ ]:
import glob, os, time
CKPT = None
HF_TOKEN_MANUAL = ""    # optional: paste token here to skip the prompt

c = glob.glob("/content/models/*.pt")
if c:
    CKPT = c[0]
    print("ckpt reused (no download):", CKPT, "|", round(os.path.getsize(CKPT)/1e9, 2), "GB")
else:
    from huggingface_hub import hf_hub_download
    from getpass import getpass
    TOK = (HF_TOKEN_MANUAL or "").strip() or None
    if not TOK:
        print(">>> Enter your Hugging Face token in the input box that appears BELOW this cell <<<")
        TOK = getpass("HF token: ").strip() or None
    if not TOK:
        raise RuntimeError("No token entered. Paste your hf_... token in the box (or into "
                           "HF_TOKEN_MANUAL) and re-run this cell.")
    os.makedirs("/content/models", exist_ok=True)
    for lk in glob.glob("/content/models/.cache/huggingface/**/*.lock", recursive=True):
        try:
            os.remove(lk)
        except OSError:
            pass
    print("Downloading vggt_omega_1b_512.pt (~4.7 GB, one-time). Progress bar below...")
    t0 = time.time()
    CKPT = hf_hub_download(repo_id="facebook/VGGT-Omega", filename="vggt_omega_1b_512.pt",
                           token=TOK, local_dir="/content/models")
    print("download finished in %.1f min -> %s" % ((time.time() - t0) / 60.0, CKPT))
print("ckpt:", CKPT)

ckpt reused (no download): /content/models/vggt_omega_1b_512.pt | 4.58 GB
ckpt: /content/models/vggt_omega_1b_512.pt


In [ ]:
# ============ CELL 3 — VIDEO INPUT + local materialization + decode probe ============
import re as _re3
import cv2 as _cv3
VIDEO_PATH_MANUAL = ""    # full Colab path, e.g. "/content/DJI_0574.MP4"
VIDEO_DRIVE_PATH  = "I:\My Drive\DJI_0574.MP4\DJI_0574.MP4"    # ANY form ok (even Windows-style): sanitized below
VIDEO_URL         = ""    # direct https link
AUTO_FIND_DRIVE   = True  # fallback: search Drive for first .mp4/.mov

def _sanitize_win(p):
    p = p.strip().replace("\\", "/")
    p = _re3.sub(r"^[A-Za-z]:", "", p)
    parts = [q for q in p.split("/") if q not in ("", ".")]
    while parts and parts[0].lower() in ("my drive", "mydrive", "drive", "content"):
        parts = parts[1:]
    return "/".join(parts)

def _drive_find(rel):
    base = "/content/drive/MyDrive"
    for c in [os.path.join("/content/drive", rel),
              os.path.join(base, rel),
              os.path.join(base, os.path.basename(rel))]:
        if c and os.path.exists(c):
            return c
    return None

VIDEO = None
if VIDEO_PATH_MANUAL and os.path.exists(VIDEO_PATH_MANUAL):
    VIDEO = VIDEO_PATH_MANUAL
    print("video from manual path:", VIDEO)
elif VIDEO_DRIVE_PATH:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("drive mount note:", e)
    VIDEO = _drive_find(_sanitize_win(VIDEO_DRIVE_PATH))
    if VIDEO:
        print("video found on Drive:", VIDEO)
    else:
        print("WARN: Drive path not found -> falling back to auto-find")
elif VIDEO_URL:
    fname = VIDEO_URL.rstrip("/").split("/")[-1].split("?")[0]
    if not fname.lower().endswith((".mp4", ".mov")):
        fname = "video.mp4"
    VIDEO = os.path.join("/content", fname)
    !wget -q -O {VIDEO} "{VIDEO_URL}"
    print("video downloaded:", VIDEO)

if VIDEO is None and AUTO_FIND_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("drive mount note:", e)
    hits = []
    base = "/content/drive/MyDrive"
    if os.path.isdir(base):
        for r, d, fs in os.walk(base):
            for f in fs:
                if f.lower().endswith((".mp4", ".mov")):
                    hits.append(os.path.join(r, f))
            if hits:
                break
    if hits:
        VIDEO = hits[0]
        print("video auto-found in Drive:", VIDEO)

if VIDEO is None:
    from google.colab import files
    print("Upload your trimmed drone video (.mp4):")
    up = files.upload()
    for name in up.keys():
        if name.lower().endswith((".mp4", ".mov")):
            VIDEO = os.path.join("/content", name); break

assert VIDEO and os.path.exists(VIDEO), "No video provided by any method."

# --- KEY FIX: copy Drive/FUSE videos to local disk; cv2 cannot decode from FUSE ---
if VIDEO.startswith("/content/drive"):
    local = os.path.join("/content", os.path.basename(VIDEO))
    if (not os.path.exists(local)) or (os.path.getsize(local) != os.path.getsize(VIDEO)):
        print("copying video Drive -> local for reliable decode...")
        import shutil
        shutil.copyfile(VIDEO, local)
    VIDEO = local
    print("video materialized locally:", VIDEO)

def _video_ok(path):
    cap = _cv3.VideoCapture(path)
    opened = cap.isOpened()
    ret = False
    n = 0
    if opened:
        ret, _ = cap.read()
        n = int(cap.get(_cv3.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return (opened and ret), n

okdec, nframes = _video_ok(VIDEO)
assert okdec, ("Video undecodable: %s — if it is H.265/HEVC, re-export as H.264 MP4 "
               "(or trim again) and retry." % VIDEO)
print("video:", VIDEO, "| size:", round(os.path.getsize(VIDEO)/1e6, 1), "MB | frames:", nframes)

<>:5: SyntaxWarning: invalid escape sequence '\M'
<>:5: SyntaxWarning: invalid escape sequence '\M'
/tmp/ipykernel_5482/565274680.py:5: SyntaxWarning: invalid escape sequence '\M'
  VIDEO_DRIVE_PATH  = "I:\My Drive\DJI_0574.MP4\DJI_0574.MP4"    # ANY form ok (even Windows-style): sanitized below


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
video found on Drive: /content/drive/MyDrive/DJI_0574.MP4/DJI_0574.MP4
copying video Drive -> local for reliable decode...
video materialized locally: /content/DJI_0574.MP4
video: /content/DJI_0574.MP4 | size: 242.0 MB | frames: 967


In [ ]:
# ============ CELL 4 — TELEMETRY INPUT (any path form accepted; never crashes on bad strings) ============
TELEM_PATH_MANUAL = ""    # e.g. "/content/drive/MyDrive/DJI_0574.SRT"
TELEM_DRIVE_PATH  = "I:\My Drive\DJI_0574.MP4\DJI_0574.SRT"    # ANY form ok (even Windows-style): sanitized below
AUTO_FIND_TELEM   = True

TELEM_CSV = None
if TELEM_PATH_MANUAL and os.path.exists(TELEM_PATH_MANUAL):
    TELEM_CSV = TELEM_PATH_MANUAL
    print("telemetry from manual path:", TELEM_CSV)
elif TELEM_DRIVE_PATH:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("drive mount note:", e)
    rel = _sanitize_win(TELEM_DRIVE_PATH)
    TELEM_CSV = _drive_find(rel)
    if TELEM_CSV is None and rel:
        # also accept the other extension in the same location
        alt = rel[:-4] + (".csv" if rel.lower().endswith(".srt") else ".srt")
        TELEM_CSV = _drive_find(alt)
    if TELEM_CSV:
        print("telemetry from Drive (sanitized):", TELEM_CSV)
    else:
        print("WARN: sanitized Drive telemetry not found:", rel, "-> falling back to auto-find")

if TELEM_CSV is None and AUTO_FIND_TELEM:
    hits = []
    base = "/content/drive/MyDrive"
    if os.path.isdir(base):
        for r, d, fs in os.walk(base):
            for f in fs:
                if f.lower().endswith((".srt", ".csv")):
                    hits.append(os.path.join(r, f))
            if hits:
                break
    if hits:
        TELEM_CSV = hits[0]
        print("telemetry auto-found in Drive:", TELEM_CSV)

if TELEM_CSV is None:
    _c = glob.glob("/content/*.srt") + glob.glob("/content/*.csv")
    if _c:
        TELEM_CSV = _c[0]
        print("telemetry from /content:", TELEM_CSV)

if TELEM_CSV is None:
    from google.colab import files
    print("Upload FlightRecord telemetry (.srt or .csv):")
    up = files.upload()
    for name in up.keys():
        low = name.lower()
        if low.endswith((".srt", ".csv")):
            TELEM_CSV = os.path.join("/content", name); break

assert TELEM_CSV and os.path.exists(TELEM_CSV), "No telemetry provided by any method."
print("telemetry:", TELEM_CSV)

<>:3: SyntaxWarning: invalid escape sequence '\M'
<>:3: SyntaxWarning: invalid escape sequence '\M'
/tmp/ipykernel_5482/1828009429.py:3: SyntaxWarning: invalid escape sequence '\M'
  TELEM_DRIVE_PATH  = "I:\My Drive\DJI_0574.MP4\DJI_0574.SRT"    # ANY form ok (even Windows-style): sanitized below


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
telemetry from Drive (sanitized): /content/drive/MyDrive/DJI_0574.MP4/DJI_0574.SRT
telemetry: /content/drive/MyDrive/DJI_0574.MP4/DJI_0574.SRT


In [ ]:
import os, sys, json, time, gc, glob, pickle
import numpy as np
import numpy.random as _nr
import cv2, torch, skimage
print("numpy", np.__version__, "| cv2", cv2.__version__, "| torch", torch.__version__)
if "/content/third_party/vggt_omega" not in sys.path:
    sys.path.insert(0, "/content/third_party/vggt_omega")
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU first."
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
DEMO_MODE = True
print("GPU:", torch.cuda.get_device_name(0),
      "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

numpy 2.1.3 | cv2 5.0.0 | torch 2.11.0+cu128
GPU: Tesla T4 | VRAM: 15.6 GB


In [ ]:
class StitchError(RuntimeError):
    pass

def procrustes_sim3(source_pts, target_pts):
    source_pts = np.asarray(source_pts, float); target_pts = np.asarray(target_pts, float)
    n, m = source_pts.shape
    if n < 3:
        raise StitchError("Need >= 3 pts, got %d" % n)
    mu_s, mu_t = source_pts.mean(0), target_pts.mean(0)
    a, b = source_pts - mu_s, target_pts - mu_t
    var_src = np.mean(np.sum(a ** 2, axis=1))
    if var_src < 1e-7:
        raise StitchError("Source zero-variance.")
    U, D, Vt = np.linalg.svd(b.T @ a / n)
    if D[-1] < 1e-7 * D[0]:
        raise StitchError("Near-collinear correspondences.")
    d = np.eye(m)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        d[-1, -1] = -1
    R = U @ d @ Vt
    s = float(np.trace(np.diag(D) @ d) / var_src)
    if not np.isfinite(s) or s <= 0:
        raise StitchError("Invalid scale %s" % s)
    return s, R, mu_t - s * (R @ mu_s)

def align_chunks_with_pixels(chunk_local, chunk_global, overlap_indices):
    src_pts, tgt_pts = [], []
    for idx in overlap_indices:
        lp = chunk_local["indices"].index(idx)
        gp = chunk_global["indices"].index(idx)
        dl = chunk_local["depth"][lp].astype(np.float32)
        dg = chunk_global["depth"][gp].astype(np.float32)
        depth_l, conf_l = (dl[..., 0], dl[..., 1]) if dl.ndim == 3 and dl.shape[-1] == 2 else (dl, np.ones_like(dl)*2.0)
        depth_g, conf_g = (dg[..., 0], dg[..., 1]) if dg.ndim == 3 and dg.shape[-1] == 2 else (dg, np.ones_like(dg)*2.0)
        K_l, K_g = chunk_local["K"][lp], chunk_global["K"][gp]
        R_l, t_l = chunk_local["R"][lp], chunk_local["t"][lp]
        R_g, t_g = chunk_global["R"][gp], chunk_global["t"][gp]
        valid = ((conf_l > 1.5) & (conf_g > 1.5) & np.isfinite(depth_l) & np.isfinite(depth_g)
                 & (depth_l > 0.05) & (depth_g > 0.05) & (depth_l < 500.0) & (depth_g < 500.0))
        if not valid.any():
            continue
        ys, xs = np.where(valid)
        if len(ys) > 2000:
            pick = np.random.choice(len(ys), 2000, replace=False)
            ys, xs = ys[pick], xs[pick]
        fx_l, fy_l, cx_l, cy_l = K_l[0,0], K_l[1,1], K_l[0,2], K_l[1,2]
        Z_l = depth_l[ys, xs]
        w_l = (np.stack([(xs-cx_l)*Z_l/fx_l, (ys-cy_l)*Z_l/fy_l, Z_l], -1) - t_l[None,:]) @ R_l
        fx_g, fy_g, cx_g, cy_g = K_g[0,0], K_g[1,1], K_g[0,2], K_g[1,2]
        Z_g = depth_g[ys, xs]
        w_g = (np.stack([(xs-cx_g)*Z_g/fx_g, (ys-cy_g)*Z_g/fy_g, Z_g], -1) - t_g[None,:]) @ R_g
        src_pts.append(w_l); tgt_pts.append(w_g)
    if not src_pts:
        raise StitchError("No valid pixel correspondences.")
    return procrustes_sim3(np.vstack(src_pts), np.vstack(tgt_pts))

def _basis(e1, up):
    e1 = np.asarray(e1, float); e1 = e1 / np.linalg.norm(e1)
    u = np.asarray(up, float)
    u = u - (u @ e1) * e1
    n = np.linalg.norm(u)
    if n < 1e-6:
        tmp = np.array([0.0, 1.0, 0.0])
        if abs(tmp @ e1) > 0.9:
            tmp = np.array([1.0, 0.0, 0.0])
        u = tmp - (tmp @ e1) * e1; n = np.linalg.norm(u)
    e3 = u / n
    e2 = np.cross(e3, e1)
    return np.stack([e1, e2, e3], axis=1)

def _sim3_ordered_fit(src, dst, src_up, dst_up):
    if len(src) < 2:
        raise StitchError("Need >= 2 points.")
    mu_s, mu_d = src.mean(0), dst.mean(0)
    a = src - mu_s
    b = dst - mu_d
    e1_s = src[-1] - src[0]
    e1_t = dst[-1] - dst[0]
    if np.linalg.norm(e1_s) < 1e-8 or np.linalg.norm(e1_t) < 1e-8:
        raise StitchError("Degenerate segment (no motion).")
    R = _basis(e1_t, dst_up) @ _basis(e1_s, src_up).T
    var_s = np.mean(np.sum(a ** 2, axis=1))
    var_t = np.mean(np.sum(b ** 2, axis=1))
    s = float(np.sqrt(var_t / var_s)) if var_s > 1e-8 else 1.0
    if not np.isfinite(s) or s <= 0:
        raise StitchError("Invalid scale %s" % s)
    return s, R, mu_d - s * (R @ mu_s)

def umeyama_ransac(src, dst, inlier_thresh_m=2.0, iters=60, seed=0, src_up=None, dst_up=None):
    src = np.asarray(src, float); dst = np.asarray(dst, float)
    n = len(src)
    if src_up is None:
        g = globals().get("out")
        if g is not None and "R" in g:
            ups = np.stack([g["R"][i].T @ np.array([0.0, -1.0, 0.0])
                            for i in range(len(g["R"]))])
            src_up = ups.mean(0)
        else:
            src_up = np.array([0.0, 0.0, 1.0])
    if dst_up is None:
        dst_up = np.array([0.0, 0.0, 1.0])
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(iters):
        idx = np.sort(rng.choice(n, max(8, n // 10), replace=False))
        try:
            s, R, t = _sim3_ordered_fit(src[idx], dst[idx], src_up, dst_up)
        except StitchError:
            continue
        res = np.linalg.norm(s * (src @ R.T) + t - dst, axis=1)
        inl = res < inlier_thresh_m
        if best is None or int(inl.sum()) > best[0]:
            best = (int(inl.sum()), inl)
    if best is None or int(best[0]) < 2:
        raise StitchError("Umeyama RANSAC: insufficient inliers.")
    _, inl = best
    s, R, t = _sim3_ordered_fit(src[inl], dst[inl], src_up, dst_up)
    res = np.linalg.norm(s * (src @ R.T) + t - dst, axis=1)
    mae = float(res[inl].mean()) if inl.any() else float("inf")
    return s, R, t, mae, inl

def check_trajectory_non_collinear(centers):
    centers = np.asarray(centers, float)
    if len(centers) < 3:
        return False, 0.0, 0.0
    traj_len = float(np.sum(np.linalg.norm(np.diff(centers, axis=0), axis=1)))
    if traj_len <= 1e-8:
        return False, 0.0, traj_len
    c = centers - centers.mean(0)
    _, _, Vt = np.linalg.svd(c)
    proj = np.outer(c @ Vt[0], Vt[0])
    max_dev = float(np.max(np.linalg.norm(c - proj, axis=1)))
    return (max_dev / traj_len) > 0.01, max_dev, traj_len

In [ ]:
import re as _re
P1_CFG = dict(target_fps=1.0, dedup_hash_threshold=4, max_keyframes=200, min_keyframes=10,
              blur_variance_min=100.0, denoise_noise_trigger=25.0, clahe_clip_limit=2.0,
              yolo_classes=(0, 2), shadow_value_max=60,
              telemetry_speed_max_mps=25.0, telemetry_jump_flag_m=50.0,
              baseline_spacing_m=(0.5, 5.0), gps_coverage_min_pct=90.0)

def _phash(gray, h=8):
    img = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(img)
    return (dct[:h, :h] > np.median(dct[:h, :h])).flatten()

def _hamming(a, b):
    return int(np.count_nonzero(a != b))

def extract_keyframes(video_path, cfg):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step_f = max(1, int(round(fps / cfg["target_fps"])))
    cands, ts, fi = [], [], 0
    while True:
        ret, fr = cap.read()
        if not ret:
            break
        if fi % step_f == 0:
            cands.append(fr); ts.append(fi / fps)
        fi += 1
    cap.release()
    kept, hashes, kept_ts = [], [], []
    for fr, t in zip(cands, ts):
        hb = _phash(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY))
        if any(_hamming(hb, ph) <= cfg["dedup_hash_threshold"] for ph in hashes):
            continue
        kept.append(fr); hashes.append(hb); kept_ts.append(t)
        if len(kept) >= cfg["max_keyframes"]:
            break
    return kept, kept_ts, fps, len(cands)

def blur_filter(frames, ts, cfg):
    out_f, out_t, dropped = [], [], 0
    for fr, t in zip(frames, ts):
        var = cv2.Laplacian(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
        if var >= cfg["blur_variance_min"]:
            out_f.append(fr); out_t.append(t)
        else:
            dropped += 1
    return out_f, out_t, dropped

def _noise_sigma(gray):
    g = gray.astype(np.float64)
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], float)
    conv = cv2.filter2D(g, -1, k)
    h, w = g.shape
    return float(np.sqrt(np.pi / 2.0) * np.abs(conv).sum() / ((h - 2) * (w - 2) * 6.0))

def conditional_denoise(frames, cfg):
    out, triggered = [], 0
    for fr in frames:
        if _noise_sigma(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)) > cfg["denoise_noise_trigger"]:
            fr = cv2.fastNlMeansDenoisingColored(fr, None, 5, 5, 7, 21)
            triggered += 1
        out.append(fr)
    return out, triggered

def apply_clahe(frames, cfg):
    clahe = cv2.createCLAHE(clipLimit=cfg["clahe_clip_limit"], tileGridSize=(8, 8))
    out = []
    for fr in frames:
        lab = cv2.cvtColor(fr, cv2.COLOR_BGR2LAB)
        lab[..., 0] = clahe.apply(lab[..., 0])
        out.append(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR))
    return out

def build_exclude_masks(frames, cfg):
    shadow = [cv2.cvtColor(f, cv2.COLOR_BGR2HSV)[..., 2] < cfg["shadow_value_max"] for f in frames]
    try:
        from ultralytics import YOLO
        model = YOLO("yolov8n-seg.pt")
        masks, mode = [], "yolov8n-seg-cpu+shadow"
        for fr, sh in zip(frames, shadow):
            m = sh.copy()
            r = model.predict(fr, device="cpu", verbose=False, classes=list(cfg["yolo_classes"]))[0]
            if r.masks is not None:
                for mm in r.masks.data:
                    m |= (mm.cpu().numpy() > 0.5)
            masks.append(m)
        return masks, mode
    except Exception as e:
        print("YOLO unavailable (%s); shadow masks only." % e)
        return shadow, "shadow-only"

def wgs84_to_enu(lat, lon, alt):
    from pyproj import Transformer
    tr = Transformer.from_crs("EPSG:4326", "EPSG:4978", always_xy=True)
    x, y, z = tr.transform(lon, lat, alt)
    X = np.stack([x, y, z], -1)
    la0, lo0 = np.radians(lat[0]), np.radians(lon[0])
    sl, cl, so, co = np.sin(la0), np.cos(la0), np.sin(lo0), np.cos(lo0)
    R = np.array([[-so, co, 0],
                  [-sl * co, -sl * so, cl],
                  [cl * co, cl * so, sl]])
    return (X - X[0]) @ R.T, X[0], R

def _srt_time_to_sec(tok):
    m = _re.match(r"(\d+):(\d+):(\d+)[,.](\d+)", tok.strip())
    if not m:
        return None
    h, mi, s, ms = m.groups()
    return int(h) * 3600 + int(mi) * 60 + int(s) + int(ms.ljust(3, "0")[:3]) / 1000.0

def parse_srt(path):
    txt = open(path, "r", encoding="utf-8", errors="ignore").read()
    re_gps = _re.compile(r"(-?\d{1,3}\.\d+)\s*,\s*(-?\d{1,3}\.\d+)\s*,\s*(-?\d+\.?\d*)")
    re_latlon = _re.compile(r"LAT\s*[:=]?\s*(-?\d+\.?\d+).*?LON\s*[:=]?\s*(-?\d+\.?\d+)", _re.I)
    re_alt = _re.compile(r"(?<![A-Za-z.])H\s*:\s*(-?\d+\.?\d*)")
    re_att = {k: _re.compile(k + r"\s*[:=]?\s*(-?\d+\.?\d*)", _re.I) for k in ("Pitch", "Roll", "Yaw")}
    rows = []
    for block in _re.split(r"\n\s*\n", txt):
        lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
        t = None
        for ln in lines:
            if "-->" in ln:
                t = _srt_time_to_sec(ln.split("-->")[0])
                break
        if t is None:
            continue
        body = " ".join(lines)
        lat = lon = alt = None
        m = re_gps.search(body)
        if m:
            lat, lon, alt = float(m.group(1)), float(m.group(2)), float(m.group(3))
        else:
            m2 = re_latlon.search(body)
            if m2:
                lat, lon = float(m2.group(1)), float(m2.group(2))
        if lat is None or lon is None:
            continue
        if alt is None:
            ma = re_alt.search(body)
            alt = float(ma.group(1)) if ma else 0.0
        att = []
        for k in ("Pitch", "Roll", "Yaw"):
            mk = re_att[k].search(body)
            att.append(float(mk.group(1)) if mk else np.nan)
        rows.append([t, lat, lon, alt] + att)
    assert rows, "SRT: no GPS rows parsed - is this a DJI telemetry .srt?"
    print("SRT parsed %d telemetry rows" % len(rows))
    return np.array(rows, dtype=np.float64)

def _finalize_telemetry(t, lat, lon, alt, cfg, attitude=None):
    t = np.asarray(t, float).copy(); lat = np.asarray(lat, float).copy()
    lon = np.asarray(lon, float).copy(); alt = np.asarray(alt, float).copy()
    bad_t = ~np.isfinite(t)
    if bad_t.any() and np.isfinite(t).sum() >= 2:
        good = np.flatnonzero(~bad_t)
        t[bad_t] = np.interp(np.flatnonzero(bad_t), good, t[good])
        print("interpolated %d malformed timestamps" % int(bad_t.sum()))
    ok = np.isfinite(t) & np.isfinite(lat) & np.isfinite(lon)
    t, lat, lon, alt = t[ok], lat[ok], lon[ok], np.nan_to_num(alt[ok])
    o = np.argsort(t); t, lat, lon, alt = t[o], lat[o], lon[o], alt[o]
    enu, X0, R = wgs84_to_enu(lat, lon, alt)
    keep = [0]
    for i in range(1, len(enu)):
        if np.linalg.norm(enu[i] - enu[keep[-1]]) <= cfg["telemetry_jump_flag_m"]:
            keep.append(i)
    keep = np.array(keep)
    enu_k = enu[keep]
    d = np.linalg.norm(np.diff(enu_k, axis=0), axis=1)
    dt = np.diff(t[keep]); dt[dt <= 0] = np.nan
    spd = float(np.nanpercentile(d / dt, 99)) if len(d) else 0.0
    from pyproj import Transformer
    tr2 = Transformer.from_crs("EPSG:4978", "EPSG:4326", always_xy=True)
    Xb = X0 + enu_k @ R
    lo2, la2, _ = tr2.transform(Xb[:, 0], Xb[:, 1], Xb[:, 2])
    rt = float(np.max(np.hypot((la2 - lat[keep]) * 111320.0,
                               (lo2 - lon[keep]) * 111320.0 * np.cos(np.radians(lat[keep])))))
    print("telemetry: rows=%d kept=%d p99_speed=%.1f m/s roundtrip=%.2e m"
          % (len(t), len(keep), spd, rt))
    if spd > cfg["telemetry_speed_max_mps"]:
        print("WARN: speed gate exceeded.")
    assert rt < 1e-3, "ENU round-trip gate failed"
    outd = {"t": t[keep], "enu": enu_k, "lat": lat[keep], "lon": lon[keep], "alt": alt[keep]}
    if attitude is not None:
        outd["attitude"] = np.asarray(attitude, dtype=np.float64)[keep]
    return outd

def parse_telemetry(path, cfg):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".srt":
        arr = parse_srt(path)
        return _finalize_telemetry(arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3], cfg,
                                   attitude=arr[:, 4:7])
    import pandas as pd
    df = pd.read_csv(path, on_bad_lines="skip")
    df.columns = [str(c).strip().lower() for c in df.columns]
    num = df.apply(lambda c: pd.to_numeric(c, errors="coerce"))
    tcol = next((c for c in df.columns if "create_time" in c or "timestamp" in c), None)
    if tcol is None:
        tcol = next((c for c in num.columns if num[c].notna().any() and num[c].dropna().min() > 1e9), None)
    latc = next((c for c in df.columns if "gps_lat" in c or c.endswith("lat") or "latitude" in c), None)
    lonc = next((c for c in df.columns if "gps_lon" in c or c.endswith("lon") or "longitude" in c), None)
    if latc is None or lonc is None:
        vary = [c for c in num.columns if num[c].notna().sum() > 1
                and (num[c].max() - num[c].min()) > 1e-7 and num[c].abs().max() <= 180]
        lats = [c for c in vary if num[c].abs().max() <= 90]
        lons = [c for c in vary if num[c].min() < -90 or num[c].max() > 90]
        latc = latc or (lats[0] if lats else None)
        lonc = lonc or (lons[0] if lons else (lats[1] if len(lats) > 1 else None))
    assert tcol and latc and lonc, "telemetry: time/lat/lon columns not found"
    print("telemetry columns:", tcol, latc, lonc)
    altc = next((c for c in df.columns if "alt" in c or "height" in c), None)
    t = num[tcol].to_numpy(float); lat = num[latc].to_numpy(float)
    lon = num[lonc].to_numpy(float)
    alt = num[altc].to_numpy(float) if altc else np.zeros_like(lat)
    return _finalize_telemetry(t, lat, lon, alt, cfg)

def telemetry_keyframe_gates(ts_key, tele, cfg):
    trel = tele["t"] - tele["t"][0]
    ts_key = np.asarray(ts_key, float)
    cov = float(np.mean((ts_key >= trel[0]) & (ts_key <= trel[-1])) * 100.0)
    pos = np.stack([np.interp(ts_key, trel, tele["enu"][:, j]) for j in range(3)], -1)
    d = np.linalg.norm(np.diff(pos, axis=0), axis=1)
    med = float(np.median(d)) if len(d) else 0.0
    return cov, med

def phase1_process(video_path, cfg):
    frs, ts, fps, n_cand = extract_keyframes(video_path, cfg)
    n_dedup = len(frs)
    frs, ts, n_blur = blur_filter(frs, ts, cfg)
    frs, n_den = conditional_denoise(frs, cfg)
    frs = apply_clahe(frs, cfg)
    rgb = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frs]
    print("PHASE1: candidates=%d dedup=%d after_blur=%d denoised=%d"
          % (n_cand, n_dedup, len(rgb), n_den))
    assert cfg["min_keyframes"] <= len(rgb) <= cfg["max_keyframes"], \
        "keyframe count %d outside [%d,%d]" % (len(rgb), cfg["min_keyframes"], cfg["max_keyframes"])
    return rgb, ts, n_cand, n_dedup, n_blur, n_den

In [ ]:
WORK_MAX = 1600

def _resize_max(fr, max_side=WORK_MAX):
    h, w = fr.shape[:2]
    if max(h, w) <= max_side:
        return fr
    sc = max_side / float(max(h, w))
    return cv2.resize(fr, (int(w * sc), int(h * sc)), interpolation=cv2.INTER_AREA)

def extract_keyframes(video_path, cfg):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step_f = max(1, int(round(fps / cfg["target_fps"])))
    cands, ts, fi = [], [], 0
    while True:
        ret, fr = cap.read()
        if not ret:
            break
        if fi % step_f == 0:
            cands.append(_resize_max(fr)); ts.append(fi / fps)
        fi += 1
    cap.release()
    kept, hashes, kept_ts = [], [], []
    for fr, t in zip(cands, ts):
        hb = _phash(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY))
        if any(_hamming(hb, ph) <= cfg["dedup_hash_threshold"] for ph in hashes):
            continue
        kept.append(fr); hashes.append(hb); kept_ts.append(t)
        if len(kept) >= cfg["max_keyframes"]:
            break
    return kept, kept_ts, fps, len(cands)

def phase1_process(video_path, cfg):
    frs, ts, fps, n_cand = extract_keyframes(video_path, cfg)
    n_dedup = len(frs)
    frs, ts, n_blur = blur_filter(frs, ts, cfg)
    frs, n_den = conditional_denoise(frs, cfg)
    frs = apply_clahe(frs, cfg)
    masks, mask_mode = build_exclude_masks(frs, cfg)
    rgb = [cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (512, 512),
                      interpolation=cv2.INTER_AREA) for f in frs]
    del frs
    gc.collect()
    print("PHASE1: candidates=%d dedup=%d after_blur=%d denoised=%d masks=%s"
          % (n_cand, n_dedup, len(rgb), n_den, mask_mode))
    assert cfg["min_keyframes"] <= len(rgb) <= cfg["max_keyframes"], \
        "keyframe count %d outside [%d,%d]" % (len(rgb), cfg["min_keyframes"], cfg["max_keyframes"])
    return rgb, ts, masks, mask_mode, {"candidates": n_cand, "dedup": n_dedup,
                                       "blur_dropped": n_blur, "denoised": n_den,
                                       "mask_mode": mask_mode}
print("RAM-safe overrides active: working res <= %d px, downstream 512 px" % WORK_MAX)

RAM-safe overrides active: working res <= 1600 px, downstream 512 px


In [ ]:
def blur_filter(frames, ts, cfg):
    out_f, out_t, dropped = [], [], 0
    for fr, t in zip(frames, ts):
        g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
        g = cv2.resize(g, (1024, 1024), interpolation=cv2.INTER_AREA)
        var = cv2.Laplacian(g, cv2.CV_64F).var()
        if var >= cfg["blur_variance_min"]:
            out_f.append(fr); out_t.append(t)
        else:
            dropped += 1
    return out_f, out_t, dropped
print("blur gate normalized to 1024px reference")

blur gate normalized to 1024px reference


In [ ]:
def load_model():
    from vggt_omega.models import VGGTOmega
    precision = "fp16" if torch.cuda.get_device_capability() >= (7, 0) else "fp32"
    model = VGGTOmega().eval()
    model.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))
    return model.cuda(), "cuda", precision

def run_windowed(model, frames, timestamps, chunk_size=6, overlap=3,
                 device="cuda", precision="fp16", ckpt_dir=None, video_name=""):
    from vggt_omega.utils.pose_enc import encoding_to_camera
    n = len(frames); IN = 512
    step = max(1, chunk_size - overlap)
    starts = list(range(0, n, step))
    records, per_chunk = [], []
    inspected = False
    overlap_errs = []
    done_starts = set()
    if ckpt_dir:
        try:
            mp = os.path.join(ckpt_dir, "ckpt.json")
            if os.path.exists(mp):
                meta = json.load(open(mp))
                if (meta.get("video") == video_name and meta.get("chunk") == chunk_size
                        and meta.get("overlap") == overlap and meta.get("n") == n):
                    for st in meta.get("completed_starts", []):
                        p = os.path.join(ckpt_dir, "chunk_%05d.pkl" % st)
                        if os.path.exists(p):
                            records.append(pickle.load(open(p, "rb")))
                            done_starts.add(st)
                    if records:
                        records.sort(key=lambda r: r["indices"][0])
                        print("RESUMED %d/%d chunks from checkpoint" % (len(records), len(starts)))
        except Exception as e:
            print("resume failed (%s); fresh run." % e)
            records, done_starts = [], set()
    t0 = time.time()
    for start in starts:
        if start in done_starts:
            continue
        end = min(start + chunk_size, n)
        sl = frames[start:end]
        tc = time.time()
        tens = []
        for img in sl:
            r = img if img.shape[:2] == (IN, IN) else cv2.resize(img, (IN, IN), interpolation=cv2.INTER_AREA)
            tens.append(torch.from_numpy(r.transpose(2, 0, 1).astype(np.float32) / 255.0))
        batch = torch.stack(tens).unsqueeze(0).cuda()
        with torch.no_grad():
            if precision == "fp16":
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    pred = model(batch)
            else:
                pred = model(batch)
        ext, intr = encoding_to_camera(pred["pose_enc"], pred["images"].shape[-2:])
        if not inspected:
            print("INSPECTION keys:", list(pred.keys()), "| ext", tuple(ext.shape),
                  "| fx", float(intr[0, 0, 0, 0]))
            inspected = True
        ext_np = ext[0].float().cpu().numpy(); intr_np = intr[0].float().cpu().numpy()
        dep = pred["depth"][0].float().cpu().numpy()
        if dep.ndim == 4 and dep.shape[-1] == 1:
            dep = dep[..., 0]
        conf = (pred["depth_conf"][0].float().cpu().numpy() if "depth_conf" in pred
                else np.ones(dep.shape, dtype=np.float32) * 2.0)
        Rs, ts, Cs, Ks, Ds = [], [], [], [], []
        for i in range(len(sl)):
            R = ext_np[i, :3, :3].astype(np.float64); t = ext_np[i, :3, 3].astype(np.float64)
            K = intr_np[i].astype(np.float64).copy()
            if K[0, 0] <= 2.0:
                K[0, 0] *= IN; K[0, 2] *= IN; K[1, 1] *= IN; K[1, 2] *= IN
            Rs.append(R); ts.append(t); Cs.append(-R.T @ t); Ks.append(K)
            Ds.append(np.stack([dep[i].astype(np.float32), conf[i].astype(np.float32)], axis=-1))
        rec = {"indices": list(range(start, end)), "R": Rs, "t": ts,
               "centers": Cs, "K": Ks, "depth": Ds}
        records.append(rec)
        if ckpt_dir:
            try:
                pickle.dump(rec, open(os.path.join(ckpt_dir, "chunk_%05d.pkl" % start), "wb"))
                done_starts.add(start)
                if len(done_starts) % 5 == 0 or start == starts[-1]:
                    json.dump({"video": video_name, "chunk": chunk_size, "overlap": overlap,
                               "n": n, "completed_starts": sorted(done_starts)},
                              open(os.path.join(ckpt_dir, "ckpt.json"), "w"))
            except Exception as e:
                print("checkpoint write skipped:", e)
        del batch, pred, ext, intr
        torch.cuda.empty_cache(); gc.collect()
        per_chunk.append(round(time.time() - tc, 3))
    records.sort(key=lambda r: r["indices"][0])
    gR, gt, gC, gK, gD, gI = [], [], [], [], [], []
    c0 = records[0]
    gR += c0["R"]; gt += c0["t"]; gC += c0["centers"]; gK += c0["K"]; gD += c0["depth"]; gI += c0["indices"]
    scales = [1.0]
    for k in range(1, len(records)):
        cur = records[k]; ci = cur["indices"]
        common = [i for i in ci if i in gI]
        if len(common) < 1:
            raise StitchError("Chunk %d zero overlap; raise overlap." % k)
        gchunk = {"indices": gI, "R": gR, "t": gt, "centers": gC, "K": gK, "depth": gD}
        s, Ra, ta = align_chunks_with_pixels(cur, gchunk, common)
        if not (0.2 < s < 5.0) or not np.all(np.isfinite([s] + list(Ra.ravel()) + list(ta))):
            raise StitchError("Degenerate stitch chunk %d: s=%s" % (k, s))
        scales.append(float(s))
        for idx in common:
            lp = ci.index(idx); tp = gI.index(idx)
            overlap_errs.append(float(np.linalg.norm(s * (Ra @ cur["centers"][lp]) + ta - gC[tp])))
        for li, idx in enumerate(ci):
            if idx in gI:
                continue
            c_new = s * (Ra @ cur["centers"][li]) + ta
            R_new = cur["R"][li] @ Ra.T
            t_new = -R_new @ c_new
            d_new = cur["depth"][li].copy(); d_new[..., 0] *= s
            gR.append(R_new); gt.append(t_new); gC.append(c_new)
            gK.append(cur["K"][li]); gD.append(d_new); gI.append(idx)
    centers_arr = np.stack(gC)
    steps = np.linalg.norm(np.diff(centers_arr, axis=0), axis=1)
    mean_step = float(steps.mean()) if len(steps) else 1.0
    disc_pct = float(100.0 * np.mean(overlap_errs) / mean_step) if overlap_errs else 0.0
    return {"R": np.stack(gR).astype(np.float32), "t": np.stack(gt).astype(np.float32),
            "centers": centers_arr.astype(np.float32), "K": np.stack(gK).astype(np.float32),
            "depth": np.stack(gD).astype(np.float16),
            "timestamps": np.array(timestamps, dtype=np.float64),
            "stitch_scales": np.array(scales, dtype=np.float32),
            "overlap_discrepancy_pct": disc_pct,
            "per_chunk_seconds": per_chunk,
            "total_seconds": round(time.time() - t0, 3)}

def export_pointcloud(R, t, K, depth, frames, out_dir, max_points=3000000):
    import trimesh
    pts, cols = [], []
    for i in range(len(R)):
        dp = depth[i].astype(np.float32)
        d, c = (dp[..., 0], dp[..., 1]) if dp.ndim == 3 and dp.shape[-1] == 2 else (dp, np.ones_like(dp)*2.0)
        h, w = d.shape
        fx, fy, cx, cy = K[i][0,0], K[i][1,1], K[i][0,2], K[i][1,2]
        img = frames[i]
        if img.shape[:2] != (h, w):
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_LINEAR)
        yg, xg = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
        ok = np.isfinite(d) & (d > 0.05) & (d < 500.0) & (c > 1.5)
        if not ok.any():
            continue
        xv, yv, dv = xg[ok], yg[ok], d[ok]
        pc = np.stack([(xv-cx)*dv/fx, (yv-cy)*dv/fy, dv], axis=-1)
        pts.append((pc - t[i][None, :]) @ R[i]); cols.append(img[ok])
    if not pts:
        raise RuntimeError("No valid points.")
    V = np.vstack(pts).astype(np.float32); C = np.vstack(cols).astype(np.uint8)
    dist = np.linalg.norm(V - V.mean(0), axis=1)
    m = dist <= np.percentile(dist, 99.0); V, C = V[m], C[m]
    if len(V) > max_points:
        ix = np.linspace(0, len(V)-1, max_points).astype(int); V, C = V[ix], C[ix]
    pcd = trimesh.PointCloud(vertices=V, colors=np.hstack([C, np.full((len(C), 1), 255, np.uint8)]))
    ply = os.path.join(out_dir, "vggt_pointcloud.ply"); pcd.export(ply)
    glb = os.path.join(out_dir, "vggt_pointcloud.glb")
    try:
        sc = trimesh.Scene(); sc.add_geometry(pcd); sc.export(glb)
        with open(glb, "rb") as f:
            assert f.read(4) == b"glTF"
    except Exception as e:
        print("GLB skipped:", e); glb = ""
    print("Kept %d points" % len(V))
    return ply, glb, len(V)

In [ ]:
def gate(name, value, limit, mode="max", demo=DEMO_MODE):
    ok = (value <= limit) if mode == "max" else (value >= limit)
    tag = "PASS" if ok else ("WARN(demo)" if demo else "FAIL")
    print("[GATE] %-28s %10.4f  limit %-10s %s" % (name, value, limit, tag))
    if not ok and not demo:
        raise RuntimeError("Gate failed: " + name)
    return bool(ok)

P1_CFG["target_fps"] = 4.0        # denser views: better averaging, fewer holes
P1_CFG["shadow_value_max"] = 40   # was 60: stop masking dark tree crowns as shadow

assert TELEM_CSV, "Phase-2 completion requires telemetry: upload the .srt in CELL 4."
frames, ts_list, EXCLUDE_MASKS, MASK_MODE, P1_REPORT = phase1_process(VIDEO, P1_CFG)
TELENU = parse_telemetry(TELEM_CSV, P1_CFG)

cov, med_spacing = telemetry_keyframe_gates(ts_list, TELENU, P1_CFG)
gate("gps_coverage_pct", cov, P1_CFG["gps_coverage_min_pct"], mode="min")
gate("baseline_spacing_m", med_spacing, P1_CFG["baseline_spacing_m"][1])
if med_spacing < P1_CFG["baseline_spacing_m"][0]:
    print("[GATE] baseline_spacing below 0.5 m (hovering?) - WARN")

CHUNK, OVERLAP = 6, 3
torch.cuda.empty_cache()
model, device, precision = load_model()
torch.cuda.reset_peak_memory_stats()
out = run_windowed(model, frames, ts_list, CHUNK, OVERLAP, device, precision,
                   ckpt_dir=os.path.join(OUT, "ckpt"), video_name=os.path.basename(VIDEO))

d_all = out["depth"][..., 0].astype(np.float32)
finite_pct = float(np.mean(np.isfinite(d_all)) * 100.0)
gate("depth_finite_pct", finite_pct, 99.0, mode="min")
gate("overlap_discrepancy_pct", out["overlap_discrepancy_pct"], 5.0)

trel = TELENU["t"] - TELENU["t"][0]
ts_a = np.asarray(ts_list, float)
enu_at_frames = np.stack([np.interp(ts_a, trel, TELENU["enu"][:, j]) for j in range(3)], -1)
s_u, R_u, t_u, mae_u, inl_u = umeyama_ransac(out["centers"].astype(np.float64), enu_at_frames)
inlier_pct = float(100.0 * inl_u.mean())
print("UMEYAMA: s=%.4f m/unit  MAE=%.2f m  inliers=%.1f%%" % (s_u, mae_u, inlier_pct))
gate("umeyama_mae_m", mae_u, 3.0)
gate("umeyama_inliers_pct", inlier_pct, 80.0, mode="min")
gate_ok_s = (0.1 <= s_u <= 100.0)
print("[GATE] scale_s_bounds               %10.4f  limit [0.1,100]  %s"
      % (s_u, "PASS" if gate_ok_s else "FAIL"))
assert gate_ok_s, "Umeyama scale outside [0.1,100]"

t_metric = out["t"].astype(np.float32) * np.float32(s_u)
centers_metric = out["centers"].astype(np.float32) * np.float32(s_u)
depth_metric = out["depth"].astype(np.float32); depth_metric[..., 0] *= np.float32(s_u)
print("fusion frame: unitless (stable) | metric copies saved for Phase 5")

ply, glb, npts = export_pointcloud(out["R"], out["t"], out["K"], out["depth"], frames, OUT)
np.savez_compressed(os.path.join(OUT, "vggt_raw.npz"),
                    R=out["R"], t=out["t"], centers=out["centers"], K=out["K"],
                    depth=out["depth"], timestamps=out["timestamps"],
                    stitch_scales=out["stitch_scales"],
                    sim3_s=np.float32(s_u), sim3_R=R_u, sim3_t=t_u,
                    source_video=os.path.basename(VIDEO))
np.savez_compressed(os.path.join(OUT, "vggt_raw_metric.npz"),
                    R=out["R"], t=t_metric, centers=centers_metric, K=out["K"],
                    depth=depth_metric.astype(np.float16), timestamps=out["timestamps"],
                    stitch_scales=out["stitch_scales"], source_video=os.path.basename(VIDEO))
json.dump({"s": s_u, "R": R_u.tolist(), "t": t_u.tolist(),
           "mae_m": mae_u, "inlier_pct": inlier_pct},
          open(os.path.join(OUT, "sim3.json"), "w"), indent=2)
json.dump(out["centers"].tolist(), open(os.path.join(OUT, "trajectory.json"), "w"), indent=2)
peak_vram = round(torch.cuda.max_memory_allocated() / 1024 ** 3, 3)
gate("peak_vram_gb", peak_vram, 14.0)
log = {"n_frames": len(frames), "per_chunk_seconds": out["per_chunk_seconds"],
       "total_seconds": out["total_seconds"], "peak_vram_gb": peak_vram,
       "device": device, "precision": precision, "torch_version": torch.__version__,
       "stitch_scales": [float(x) for x in out["stitch_scales"]],
       "overlap_discrepancy_pct": out["overlap_discrepancy_pct"],
       "depth_finite_pct": finite_pct,
       "umeyama": {"s": s_u, "mae_m": mae_u, "inlier_pct": inlier_pct},
       "phase1": P1_REPORT, "gps_coverage_pct": cov, "baseline_spacing_m": med_spacing,
       "telemetry_rows": len(TELENU["t"])}
json.dump(log, open(os.path.join(OUT, "run_log.json"), "w"), indent=2)
ok_nc, dev, tlen = check_trajectory_non_collinear(out["centers"])
print("=" * 60)
print("stitch scales :", log["stitch_scales"])
print("trajectory    : len=%.2f dev=%.3f (%s)" % (tlen, dev, "NON-COLLINEAR" if ok_nc else "COLLINEAR"))
print("metric scale  : %.4f m/unit (metric npz only; fusion unitless)" % s_u)
print("=" * 60)

PHASE1: candidates=139 dedup=45 after_blur=45 denoised=0 masks=yolov8n-seg-cpu+shadow
SRT parsed 1075 telemetry rows
telemetry: rows=1075 kept=1075 p99_speed=368.6 m/s roundtrip=8.40e-10 m
WARN: speed gate exceeded.
[GATE] gps_coverage_pct               100.0000  limit 90.0       PASS
[GATE] baseline_spacing_m              11.0546  limit 5.0        WARN(demo)
INSPECTION keys: ['camera_and_register_tokens', 'pose_enc', 'depth', 'depth_conf', 'images'] | ext (1, 6, 3, 4) | fx 676.9404907226562
[GATE] depth_finite_pct               100.0000  limit 99.0       PASS
[GATE] overlap_discrepancy_pct         21.3008  limit 5.0        WARN(demo)
UMEYAMA: s=57.2749 m/unit  MAE=1.26 m  inliers=6.7%
[GATE] umeyama_mae_m                    1.2596  limit 3.0        PASS
[GATE] umeyama_inliers_pct              6.6667  limit 80.0       WARN(demo)
[GATE] scale_s_bounds                  57.2749  limit [0.1,100]  PASS
fusion frame: unitless (stable) | metric copies saved for Phase 5
Kept 3000000 points
[GA

In [ ]:
import trimesh
from skimage import measure

CONF_MIN, VOX_TARGET_DIM, TRUNC_MULT = 1.5, 512, 7.0
MASKS_EXCLUDE = [cv2.resize(m.astype(np.uint8), (512, 512),
                            interpolation=cv2.INTER_NEAREST).astype(bool) for m in EXCLUDE_MASKS]

def _dc(dp):
    dp = dp.astype(np.float32)
    if dp.ndim == 3 and dp.shape[-1] == 2:
        return dp[..., 0], dp[..., 1]
    return dp, np.ones_like(dp, dtype=np.float32) * 2.0

pts = []
for i in range(len(out["R"])):
    d, c = _dc(out["depth"][i]); h, w = d.shape
    K4 = out["K"][i]
    fx, fy, cx, cy = K4[0, 0], K4[1, 1], K4[0, 2], K4[1, 2]
    ys, xs = np.mgrid[0:h:8, 0:w:8]
    ok = np.isfinite(d[ys, xs]) & (d[ys, xs] > 0.05) & (c[ys, xs] > CONF_MIN)
    if not ok.any():
        continue
    xv, yv, dv = xs[ok], ys[ok], d[ys, xs][ok]
    pc = np.stack([(xv - cx) * dv / fx, (yv - cy) * dv / fy, dv], -1)
    pts.append((pc - out["t"][i][None, :]) @ out["R"][i])
if not pts:
    raise RuntimeError("TSDF: no valid depth points.")
P = np.vstack(pts)
pmin, pmax = P.min(0), P.max(0)
pad = 0.02 * float((pmax - pmin).max())
pmin = pmin - pad; pmax = pmax + pad
vox = float((pmax - pmin).max()) / VOX_TARGET_DIM
trunc = TRUNC_MULT * vox
dims = np.ceil((pmax - pmin) / vox).astype(int) + 1
del pts
print("voxel=%.4f trunc=%.4f grid=%s (~%.2f GB)" % (vox, trunc, tuple(dims), dims.prod() * 20 / 1e9))

tsdf = np.ones(tuple(dims), np.float32)
wvol = np.zeros(tuple(dims), np.float32)
cvol = np.zeros(tuple(dims) + (3,), np.float32)
GX, GY, GZ = np.meshgrid(np.arange(dims[0]), np.arange(dims[1]), np.arange(dims[2]), indexing="ij")
WORLD = np.stack([GX.ravel(), GY.ravel(), GZ.ravel()], 1).astype(np.float32)
WORLD = WORLD * np.float32(vox) + pmin.astype(np.float32)[None, :]
t_flat = tsdf.ravel(); w_flat = wvol.ravel(); c_flat = cvol.reshape(-1, 3)

t0 = time.time()
for i in range(len(out["R"])):
    d, c = _dc(out["depth"][i]); h, w = d.shape
    R = out["R"][i].astype(np.float32); tv = out["t"][i].astype(np.float32)
    K4 = out["K"][i]
    fx, fy, cx, cy = K4[0, 0], K4[1, 1], K4[0, 2], K4[1, 2]
    Pc = WORLD @ R.T + tv[None, :]
    Z = Pc[:, 2]
    vis = Z > 1e-4
    Zs = np.where(vis, Z, 1.0)
    u = np.where(vis, fx * Pc[:, 0] / Zs + cx, -1.0)
    v = np.where(vis, fy * Pc[:, 1] / Zs + cy, -1.0)
    ui = np.round(u).astype(np.int64); vi = np.round(v).astype(np.int64)
    in_img = vis & (ui >= 0) & (ui < w) & (vi >= 0) & (vi < h)
    uic = np.clip(ui, 0, w - 1); vic = np.clip(vi, 0, h - 1)
    dmeas = d[vic, uic]; cmeas = c[vic, uic]
    okv = in_img & np.isfinite(dmeas) & (dmeas > 0.05) & (dmeas < 500.0) & (cmeas > CONF_MIN)
    okv = okv & ~MASKS_EXCLUDE[i][vic, uic]
    sdf = dmeas - Z
    band = okv & (sdf > -trunc) & (sdf < trunc)
    idx = np.flatnonzero(band)
    if idx.size == 0:
        continue
    wold = w_flat[idx]
    wnew = wold + 1.0
    t_flat[idx] = (t_flat[idx] * wold + sdf[idx]) / wnew
    w_flat[idx] = wnew
    col = frames[i]
    if col.shape[:2] != (h, w):
        col = cv2.resize(col, (w, h), interpolation=cv2.INTER_LINEAR)
    rgb = col[vic[idx], uic[idx]].astype(np.float32)
    c_flat[idx] = (c_flat[idx] * wold[:, None] + rgb) / wnew[:, None]
obs = int((wvol > 0).sum())
print("integrated %d frames in %.1fs; observed voxels=%d" % (len(out["R"]), time.time() - t0, obs))
if obs < 1000:
    raise RuntimeError("TSDF: too few observed voxels.")
print("TSDF fusion complete -> run CELL 11B")

voxel=0.0112 trunc=0.0782 grid=(np.int64(80), np.int64(232), np.int64(513)) (~0.19 GB)
integrated 45 frames in 41.8s; observed voxels=266115
TSDF fusion complete -> run CELL 11B


In [ ]:
vol = tsdf.copy()
vol[wvol == 0] = 1.0
verts, faces, normals, _ = measure.marching_cubes(vol, level=0.0, spacing=(vox, vox, vox))
verts = verts + pmin[None, :]
vidx_v = np.clip(np.round((verts - pmin[None, :]) / vox).astype(int), 0, dims - 1)
w_at_v = wvol[vidx_v[:, 0], vidx_v[:, 1], vidx_v[:, 2]]
keep_tri = np.all(w_at_v[faces] >= 3, axis=1)
FACES_RAW = int(len(faces))
print("triangles: %d -> %d after weight mask" % (FACES_RAW, int(keep_tri.sum())))
faces = faces[keep_tri]
del tsdf, vol
gc.collect()
mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
try:
    comps = mesh.split(only_watertight=False)
    if len(comps) > 1:
        mesh = max(comps, key=lambda m: len(m.faces))
        print("kept largest component of", len(comps))
except Exception as e:
    print("component cleanup skipped:", e)
print("STEP 10 done: %d verts, %d faces -> run 11C" % (len(mesh.vertices), len(mesh.faces)))

triangles: 423402 -> 171245 after weight mask
kept largest component of 1924
STEP 10 done: 81219 verts, 152006 faces -> run 11C


In [ ]:
import subprocess as _sp
try:
    import fast_simplification  # noqa   # trimesh 4.x decimation backend
except ImportError:
    _sp.run([sys.executable, "-m", "pip", "install", "-q", "fast_simplification"])

TARGET_FACES, MAX_FACES = 150_000, 150_000   # spec hard cap; keeps max detail for demo
bbox_before = mesh.bounds.copy()
n0 = len(mesh.faces)

def _decimate(m, target):
    """Version-proof QECD: try every trimesh signature, old and new."""
    fn = getattr(m, "simplify_quadric_decimation", None)
    if fn is None:
        return m
    for kw in ({"face_count": target},
               {"number_of_faces": target},
               {"target_number_of_triangles": target},
               {"number_triangles": target}):
        try:
            return fn(**kw)
        except TypeError:
            continue
        except Exception as e:
            print("decimation attempt failed:", e)
            return m
    try:
        return fn(target)                     # positional (older trimesh)
    except TypeError:
        pass
    except Exception as e:
        print("decimation attempt failed:", e)
        return m
    try:
        import inspect
        if "percent" in inspect.signature(fn).parameters:
            return fn(percent=min(1.0, target / max(1, len(m.faces))))
    except Exception as e:
        print("decimation attempt failed:", e)
    return m

if n0 > TARGET_FACES:
    m2 = _decimate(mesh, TARGET_FACES)
    if m2 is not None and 0 < len(m2.faces) < n0:
        mesh = m2
print("faces: %d -> %d after QECD decimation" % (n0, len(mesh.faces)))

areas = mesh.area_faces
DEGEN = int((areas < 1e-12).sum())
if DEGEN:
    mesh.update_faces(areas > 1e-12)
    mesh.remove_unreferenced_vertices()
print("degenerate faces removed:", DEGEN)

gate("faces_hard_cap", float(len(mesh.faces)), float(MAX_FACES))
if len(mesh.faces) > MAX_FACES:
    if DEMO_MODE:
        print("WARN: face cap exceeded and decimator unavailable; continuing in demo mode.")
    else:
        raise RuntimeError("face cap violated")
print("STEP 11 done -> run 11D")

faces: 152006 -> 150000 after QECD decimation
degenerate faces removed: 5
[GATE] faces_hard_cap               149995.0000  limit 150000.0   PASS
STEP 11 done -> run 11D


In [ ]:
try:
    trimesh.smoothing.filter_taubin(mesh, lamb=0.4, nu=-0.42, iterations=5)  # was 0.5/-0.53 x10
except Exception as e:
    print("smoothing skipped:", e)
ext_b = bbox_before[1] - bbox_before[0]
ext_a = mesh.bounds[1] - mesh.bounds[0]
shrink = float(np.max(np.abs(ext_a - ext_b) / ext_b)) * 100.0
gate("taubin_shrink_pct", shrink, 2.0)
vidx_m = np.clip(np.round((mesh.vertices - pmin[None, :]) / vox).astype(int), 0, dims - 1)
vcols = cvol[vidx_m[:, 0], vidx_m[:, 1], vidx_m[:, 2]].astype(np.uint8)
mesh.visual.vertex_colors = np.hstack([vcols, np.full((len(vcols), 1), 255, np.uint8)])
mesh.export(os.path.join(OUT, "tsdf_mesh.ply"))
mesh.export(os.path.join(OUT, "tsdf_mesh.glb"))
print("FINAL mesh: %d verts, %d faces, watertight=%s"
      % (len(mesh.vertices), len(mesh.faces), mesh.is_watertight))
print("STEP 12 done -> run 11E")

[GATE] taubin_shrink_pct                0.7145  limit 2.0        PASS
FINAL mesh: 80079 verts, 149995 faces, watertight=False
STEP 12 done -> run 11E
